# LGM PixelDataset Smoke Test: Flowers102

Run this notebook in Google Colab with a GPU runtime selected. It clones the `generator` branch, runs the existing LGM implementation with `flowers102`, `dinov2_vitb`, PixelDataset, Adam, and `iterations=3`, then saves the metrics and synthetic dataset to mounted Google Drive.

The repository loop uses `range(0, iterations + 1)`, so `iterations=3` executes step indices 0 through 3 (four updates). This notebook preserves that behavior; it does not change the loss or optimizer algorithm. A 3-iteration smoke run is not a meaningful final accuracy evaluation.

In [ ]:
from pathlib import Path
import glob
import json
import os
import subprocess
import sys
import time
from datetime import datetime

REPO_URL = 'https://github.com/daringT-S/LGM_generator.git'
BRANCH = 'generator'
REPO_DIR = Path('/content/LGM_generator')

if REPO_DIR.exists():
    status = subprocess.check_output(['git', '-C', str(REPO_DIR), 'status', '--porcelain'], text=True)
    if status.strip():
        raise RuntimeError(f'Repository has local changes; refusing to overwrite them:\n{status}')
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)

SRC_DIR = REPO_DIR / 'src'
GIT_COMMIT = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('Repository:', REPO_DIR)
print('Branch:', BRANCH)
print('Commit:', GIT_COMMIT)

In [ ]:
import torch

print('PyTorch:', torch.__version__)
print('PyTorch CUDA build:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
print('CUDA device count:', torch.cuda.device_count())
if torch.cuda.is_available():
    for index in range(torch.cuda.device_count()):
        print(f'GPU {index}: {torch.cuda.get_device_name(index)}')
subprocess.run(['nvidia-smi'], check=False)
if not torch.cuda.is_available() or torch.cuda.device_count() < 1:
    raise RuntimeError('No CUDA GPU is visible. In Colab choose Runtime > Change runtime type > GPU, then rerun this notebook.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUTPUT_ROOT = Path('/content/drive/MyDrive/LGM_experiments')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Persistent output root:', OUTPUT_ROOT)

In [ ]:
# Keep Colab's CUDA-enabled torch/torchvision; install the remaining repository requirements.
requirements = (REPO_DIR / 'requirements.txt').read_text().splitlines()
filtered_requirements = [
    line for line in requirements
    if line.strip() and line.strip().lower() not in {'torch', 'torchvision'}
]
filtered_file = Path('/content/lgm-colab-requirements.txt')
filtered_file.write_text('\n'.join(filtered_requirements) + '\n')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(filtered_file)], check=True)

import torchvision
print('PyTorch:', torch.__version__, '| CUDA build:', torch.version.cuda)
print('torchvision:', torchvision.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('CUDA became unavailable after dependency installation; stop before training.')

In [ ]:
# The package imports are resolved from src, as required by the repository layout.
help_result = subprocess.run(
    [sys.executable, '-m', 'distillation.distill', '--help'],
    cwd=SRC_DIR, capture_output=True, text=True, check=True,
)
help_text = help_result.stdout + help_result.stderr
assert '--distill_opt' in help_text, 'This checkout does not expose --distill_opt; verify the branch/commit.'
assert '--distill_mode' in help_text, 'This checkout does not expose --distill_mode.'
print('CLI exposes both Pixel mode and optimizer options:')
print('\n'.join(line for line in help_text.splitlines() if 'distill_opt' in line or 'distill_mode' in line))

In [ ]:
DATA_ROOT = SRC_DIR / 'data' / 'datasets'
FLOWERS_ROOT = DATA_ROOT / 'flowers102'
FLOWERS_ROOT.mkdir(parents=True, exist_ok=True)

train_split = torchvision.datasets.Flowers102(root=str(FLOWERS_ROOT), split='train', download=True)
test_split = torchvision.datasets.Flowers102(root=str(FLOWERS_ROOT), split='test', download=True)
print('Flowers102 train images:', len(train_split))
print('Flowers102 test images:', len(test_split))
print('Dataset root:', FLOWERS_ROOT)

## Run the 3-iteration smoke test

The notebook records the scalar returned by the unchanged `match_gradients()` method after each update. `WANDB_MODE=offline` avoids requiring a W&B login; the LGM artifacts and a JSON metrics summary are written to Google Drive. The DINOv2 weights are downloaded through `torch.hub` on first use.

In [ ]:
os.environ['WANDB_MODE'] = 'offline'
os.chdir(SRC_DIR)
torch.manual_seed(3407)

RUN_TAG = 'flowers102_pixel_adam_it3_' + datetime.now().strftime('%Y%m%d_%H%M%S')
JOB_TAG = str(OUTPUT_ROOT / RUN_TAG)
ARGS = [
    '--dataset=flowers102',
    '--model=dinov2_vitb',
    '--distill_mode=pixel',
    '--distill_opt=adam',
    '--iterations=3',
    f'--job_tag={JOB_TAG}',
    f'--data_root={DATA_ROOT}',
    '--num_workers=2',
    '--augs_per_batch=1',
    '--checkpoint_it=100',
    '--image_log_it=500',
    '--skip_if_exists=False',
]

sys.argv = ['distill.py', *ARGS]
from config import DistillCfg
cfg = DistillCfg(explicit_bool=True).parse_args()

import distillation.distill as distill_module
LOSS_HISTORY = []
OriginalLinearGM = distill_module.LinearGM
class RecordingLinearGM(OriginalLinearGM):
    def match_gradients(self):
        loss = super().match_gradients()
        LOSS_HISTORY.append({'step': int(self.global_step), 'loss': float(loss)})
        return loss

distill_module.LinearGM = RecordingLinearGM
started = time.perf_counter()
try:
    distill_module.main(cfg)
finally:
    elapsed_seconds = time.perf_counter() - started
    distill_module.LinearGM = OriginalLinearGM

print('Configured iterations:', cfg.iterations)
print('Recorded update steps:', len(LOSS_HISTORY))
print('Loss history:', LOSS_HISTORY)
print(f'Elapsed time: {elapsed_seconds:.2f} seconds')

In [ ]:
data_candidates = list((OUTPUT_ROOT / RUN_TAG).glob('flowers102/dinov2_vitb/*/data.pth'))
if not data_candidates:
    raise FileNotFoundError(f'No distilled data.pth found under {OUTPUT_ROOT / RUN_TAG}')
data_path = max(data_candidates, key=lambda path: path.stat().st_mtime)
run_dir = data_path.parent
metrics_path = run_dir / 'baseline_metrics.json'
metrics = {
    'dataset': 'flowers102',
    'model': 'dinov2_vitb',
    'distill_mode': 'pixel',
    'distill_opt': 'adam',
    'configured_iterations': 3,
    'repository_iteration_semantics': 'steps 0..iterations inclusive',
    'recorded_update_steps': len(LOSS_HISTORY),
    'loss_history': LOSS_HISTORY,
    'elapsed_seconds': elapsed_seconds,
    'hyperparameters': {
        'ipc': cfg.ipc, 'lr': cfg.lr, 'augs_per_batch': cfg.augs_per_batch,
        'syn_res': cfg.syn_res, 'real_res': cfg.real_res, 'crop_res': cfg.crop_res,
        'init_mode': cfg.init_mode, 'aug_mode': cfg.aug_mode,
        'decorrelate_color': cfg.decorrelate_color, 'num_workers': cfg.num_workers,
    },
    'git_commit': GIT_COMMIT,
    'output_dir': str(run_dir),
    'data_path': str(data_path),
}
metrics_path.write_text(json.dumps(metrics, indent=2) + '\n')
print('Loss values:', [item['loss'] for item in LOSS_HISTORY])
print(f'Elapsed time: {elapsed_seconds:.2f} seconds')
print('Synthetic dataset:', data_path)
print('Metrics JSON:', metrics_path)